## Setup

First, install the package in development mode (run this once):
```bash
pip install -e .
```

In [ ]:
import os
from pathlib import Path

# Set your API key (or set OPENROUTER_API_KEY environment variable)
# os.environ["OPENROUTER_API_KEY"] = "your-api-key-here"

## 1. Loading a PDF Document

Load a PDF and explore its contents.

In [ ]:
from gg_qa import PDFDocument

# Replace with your PDF path
PDF_PATH = "path/to/your/document.pdf"

# Load the document
# doc = PDFDocument.from_file(PDF_PATH)
# print(f"Loaded document with {doc.page_count} pages")
# print(f"Metadata: {doc.metadata}")

In [ ]:
# Explore page content
# page = doc.get_page(1)
# print(f"Page 1 ({page.word_count} words):")
# print(page.text[:500] + "...")

## 2. Text Search

Search for text within the PDF using exact, regex, or fuzzy matching.

In [ ]:
# Exact text search
# results = doc.search_exact("important term")
# for r in results:
#     print(f"Found on page {r.page.page_number} (score: {r.score})")

In [ ]:
# Fuzzy text search (finds approximate matches)
# fuzzy_results = doc.search_fuzzy("aproximate match", threshold=60)
# for r in fuzzy_results:
#     print(f"Page {r.page.page_number} (score: {r.score:.1f}): '{r.matched_text}'")

In [ ]:
# Regex search
# regex_results = doc.search_regex(r"\d{4}-\d{2}-\d{2}")  # Find dates
# for r in regex_results:
#     print(f"Page {r.page.page_number}: '{r.matched_text}'")

## 3. Vector Embeddings

Generate embeddings for semantic search.

In [ ]:
from gg_qa import EmbeddingModel, EmbeddingCache, MockEmbeddingModel
from gg_qa.embeddings import CachedEmbeddingModel

# Use mock embeddings for testing without API calls
mock_embedder = MockEmbeddingModel(dimension=384)

# Or use real embeddings (requires API key)
# embedder = EmbeddingModel()

# With caching
# cache = EmbeddingCache(cache_dir=".embedding_cache")
# cached_embedder = CachedEmbeddingModel(embedder, cache)

In [ ]:
# Generate embeddings for document pages
# embedding_result = mock_embedder.embed_pages(doc.pages)
# print(f"Generated {len(embedding_result)} embeddings")
# print(f"Embedding dimension: {embedding_result.dimension}")

## 4. Vector Store & Semantic Search

Store embeddings and perform similarity search.

In [ ]:
from gg_qa import VectorStore

# Create vector store
store = VectorStore()

# Add document embeddings
# store.add(doc.pages, embedding_result.embeddings)
# print(f"Vector store contains {len(store)} pages")

In [ ]:
# Semantic search
# query = "What is the main conclusion?"
# query_embedding = mock_embedder.embed_text(query)
# 
# results = store.search(query_embedding, k=3)
# for r in results:
#     print(f"Page {r.page.page_number} (score: {r.score:.4f})")
#     print(f"  Preview: {r.page.text[:200]}...\n")

## 5. Hybrid Search

Combine vector and text search for better results.

In [ ]:
from gg_qa.vector_store import HybridSearcher

# Create hybrid searcher
# hybrid = HybridSearcher(store, vector_weight=0.7)
# 
# query = "conclusions and recommendations"
# query_embedding = mock_embedder.embed_text(query)
# 
# results = hybrid.search(query_embedding, query, k=3)
# for r in results:
#     print(f"Page {r.page.page_number} (combined score: {r.score:.4f})")

## 6. Question Answering with LLM

Use retrieval-augmented generation to answer questions.

In [ ]:
from gg_qa import QAEngine

# Initialize QA engine (requires OPENROUTER_API_KEY)
# engine = QAEngine(
#     vector_store=store,
#     document=doc,
#     embedding_model=mock_embedder,
#     llm_model="openai/gpt-4o-mini"
# )

In [ ]:
# Ask a question
# result = await engine.query("What are the main findings of this document?")
# 
# print("Answer:")
# print(result.answer)
# print(f"\nSources: Pages {[p.page_number for p in result.source_pages]}")

In [ ]:
# Query specific pages
# result = await engine.query_with_pages(
#     "Summarize this section",
#     page_numbers=[1, 2, 3]
# )
# print(result.answer)

## 7. Multi-Document Queries

Query across multiple related documents.

In [ ]:
from gg_qa.qa_engine import MultiDocQAEngine

# Load multiple documents
# documents = {
#     "main_report": PDFDocument.from_file("report.pdf"),
#     "appendix": PDFDocument.from_file("appendix.pdf"),
# }
# 
# # Create stores for each
# stores = {}
# for name, doc in documents.items():
#     embeddings = mock_embedder.embed_pages(doc.pages)
#     store = VectorStore()
#     store.add(doc.pages, embeddings.embeddings)
#     stores[name] = store
# 
# # Query across all documents
# multi_engine = MultiDocQAEngine(
#     documents=documents,
#     vector_stores=stores,
#     embedding_model=mock_embedder
# )
# 
# result = await multi_engine.query("Compare the findings across documents")
# print(result.answer)

## 8. Working with Embedding Cache

Cache embeddings to disk to avoid re-computing.

In [ ]:
# Create a cached embedding model
# cache = EmbeddingCache(cache_dir=".my_cache")
# cached_model = CachedEmbeddingModel(mock_embedder, cache)
# 
# # First call computes and caches
# result1 = cached_model.embed_pages(doc.pages, document_path=str(doc.path))
# print(f"First call: computed {len(result1)} embeddings")
# 
# # Second call retrieves from cache
# result2 = cached_model.embed_pages(doc.pages, document_path=str(doc.path))
# print(f"Second call: retrieved {len(result2)} embeddings from cache")

## Complete Example

Putting it all together with a real PDF.

In [ ]:
async def qa_pipeline(pdf_path: str, question: str):
    """
    Complete QA pipeline from PDF to answer.
    
    Args:
        pdf_path: Path to the PDF file
        question: Question to answer
        
    Returns:
        QAResult with answer
    """
    from gg_qa import PDFDocument, VectorStore, QAEngine
    from gg_qa.embeddings import EmbeddingModel, EmbeddingCache, CachedEmbeddingModel
    
    # Load document
    print(f"Loading PDF: {pdf_path}")
    doc = PDFDocument.from_file(pdf_path)
    print(f"  {doc.page_count} pages loaded")
    
    # Setup embeddings with caching
    print("Setting up embeddings...")
    base_embedder = EmbeddingModel()
    cache = EmbeddingCache()
    embedder = CachedEmbeddingModel(base_embedder, cache)
    
    # Generate embeddings
    print("Generating embeddings...")
    embedding_result = embedder.embed_pages(doc.pages, document_path=pdf_path)
    print(f"  {len(embedding_result)} embeddings generated")
    
    # Build vector store
    print("Building vector store...")
    store = VectorStore()
    store.add(doc.pages, embedding_result.embeddings)
    
    # Setup QA engine
    print("Initializing QA engine...")
    engine = QAEngine(
        vector_store=store,
        document=doc,
        embedding_model=base_embedder
    )
    
    # Answer question
    print(f"\nQuestion: {question}")
    result = await engine.query(question)
    
    print(f"\nAnswer:\n{result.answer}")
    print(f"\nSources: Pages {[p.page_number for p in result.source_pages]}")
    
    return result

# Usage:
# result = await qa_pipeline("document.pdf", "What is the main conclusion?")